In [1]:
METHOD = "DIRICHLET"

# Dirichlet search algorithm

In [2]:
import os
import io
import random
import copy
import torch
import json
import pickle
import contextlib
import numpy as np
from operator import itemgetter
from ultralytics import YOLO 
from mylib import myutils
from mylib import simsettings
from mylib import simtools
from mylib import yolo_patch_softmax as _
from mylib import probtools
from constants import OBS_SCALE, CELL_SIDE, MAP_RESOLUTION, DISPLAY_STEP
from constants import AGENT_HEIGHT, AGENT_RADIUS 
from constants import MAX_ITER_COEF, CONFIDENCE_THRESHOLD, LOCATION_ERROR_THRESHOLD, PSEUDO_COUNT_THRESHOLD
from constants import NUM_CLASSES, DIRICHLET_PRIOR
from constants import ACTIONS
from constants import NUM_EPOCHS
from dotenv import load_dotenv
load_dotenv()

import habitat_sim
import habitat_sim.nav as nav
from habitat.utils.visualizations import maps
from habitat_sim.utils import common as utils

# Reload imported modules
%load_ext autoreload
%autoreload 2

# Load YOLO model
yolo_model = YOLO("yolo11x.pt")

# Initialize cuda:0 device
device = "cuda:0" if torch.cuda.is_available() else "cpu"
with contextlib.redirect_stdout(io.StringIO()):
    yolo_model = yolo_model.to(device)

In [ ]:
# Load the JSON file for simulation
with open('simulation-data/simulations.json', 'r') as f:
    simulations = json.load(f)

# Simulation configuration 
simulation = simulations[27]
 
# Access its fields
SCENE = simulation["scene"]
TARGET_OBJECT = simulation["target_object"]
TARGET_OBJECT_ID = simulation["target_object_id"]
REAL_TARGET_LOCATION = simulation["target_object_location"]
INDEX = simulation["index"]

# Load the JSON file for RGB camera intrinsics
with open('simulation-data/camera-intrinsics.json', 'r') as f:
    intrinsics = json.load(f)

# Load bins per class
with open('simulation-data/object-classes-bins.json', 'r') as f:
    classes_bins = json.load(f)

# Load indoor objects
with open('simulation-data/indoor-objects.json', 'r') as f:
    data = json.load(f)
    indoor_objects = [list(item.values())[0] for item in data["indoor_classes"]]
    TARGET_OBJECT_CLASS_ID = indoor_objects.index(TARGET_OBJECT)

# Load Dirichlet priors
with open('simulation-data/dirichlet-alpha-priors-augmented.pkl', 'rb') as f:
    dirichlet_priors = pickle.load(f)

# Simulator configuration
dataset_config_file = os.path.join(os.getenv("AI2THOR_DATA"), "ai2thor-hab.scene_dataset_config.json")
sim_settings = {
    "seed": 1,
    "dataset": dataset_config_file,  # Scene dataset
    "scene": SCENE,  # Scene path
    "width": 1024,  # Spatial resolution of the observations
    "height": int(1024*OBS_SCALE),
    "default_agent": 0,
    "sensor_height": AGENT_HEIGHT,  # Height of sensors in meters
    "color_sensor": True,  # RGB sensor
    "depth_sensor": True,  # Depth sensor
    "enable_physics": False,  # kinematics only
}
# Initialize the simulator
cfg = simsettings.make_cfg(sim_settings)
sim = habitat_sim.Simulator(cfg)

In [5]:
# Get the root node of the active scene graph
scene_root = sim.get_active_scene_graph().get_root_node()
scene_bb = scene_root.cumulative_bb
scene_dims = scene_bb.size()
scene_height = scene_dims[1]  # Height of the scene

# Define navmesh settings
navmesh_settings = habitat_sim.NavMeshSettings()
navmesh_settings.agent_height = AGENT_HEIGHT
navmesh_settings.agent_radius = AGENT_RADIUS
navmesh_settings.agent_max_climb = 0.2
navmesh_settings.agent_max_slope = 45.0
navmesh_settings.include_static_objects = True  # Include static objects in the navmesh computation

# Recompute the navmesh for the current scene
sim.recompute_navmesh(sim.pathfinder, navmesh_settings)

# Generate the topdown map --> 1 cm per pixel
topdown_map = maps.get_topdown_map(sim.pathfinder, height=0, meters_per_pixel=MAP_RESOLUTION, draw_border=True)
topdown_map = myutils.map_to_rgb(topdown_map)
topdown_map = myutils.add_axis_to_map(topdown_map)
topdown_map = myutils.retain_largest_white_chunk(topdown_map)
topdown_resolution = topdown_map.shape[:2]

# Generate the grid map --> 30 cm per pixel (robot has radius 15 cm)
grid_map = maps.get_topdown_map(sim.pathfinder, height=0, meters_per_pixel=CELL_SIDE, draw_border=False)
grid_map = myutils.map_to_rgb(grid_map)
grid_map = myutils.retain_largest_white_chunk(grid_map)
grid_resolution = grid_map.shape[:2]

# Free positions
grid_free_cells, map_free_cells, world_free_coords = [], [], []

# Occupied positions
grid_occ_cells, map_occ_cells, world_occ_coords = [], [], []

# Compute free and occupied
for grid_x in range(grid_resolution[0]):
    for grid_y in range(grid_resolution[1]):
        # Convert grid coordinates to real world coordinates and then to top-down map coordinates
        real_world_z, real_world_x = maps.from_grid(grid_x, grid_y, grid_resolution, pathfinder=sim.pathfinder)
        map_x, map_y = maps.to_grid(real_world_z, real_world_x, topdown_resolution, pathfinder=sim.pathfinder)

        if not sim.pathfinder.is_navigable([real_world_x, 0.0, real_world_z]) or not topdown_map[map_x, map_y, 0] == 255 or not grid_map[grid_x, grid_y, 0] == 255:
            grid_map[grid_x, grid_y, :] = [128, 128, 128]  # Grey out non-navigable

# Filter again
grid_map = myutils.retain_largest_white_chunk(grid_map)

# Compute free and occupied
for grid_x in range(grid_resolution[0]):
    for grid_y in range(grid_resolution[1]):
        # Convert grid coordinates to real world coordinates and then to top-down map coordinates
        real_world_z, real_world_x = maps.from_grid(grid_x, grid_y, grid_resolution, pathfinder=sim.pathfinder)
        map_x, map_y = maps.to_grid(real_world_z, real_world_x, topdown_resolution, pathfinder=sim.pathfinder)

        if sim.pathfinder.is_navigable([real_world_x, 0.0, real_world_z]) and topdown_map[map_x, map_y, 0] == 255 and grid_map[grid_x, grid_y, 0] == 255:
            world_free_coords.append([real_world_x, 0.0, real_world_z])
            map_free_cells.append([map_x, map_y])
            grid_free_cells.append([grid_x, grid_y])
        else:
            world_occ_coords.append([real_world_x, 0.0, real_world_z])
            map_occ_cells.append([map_x, map_y])
            grid_occ_cells.append([grid_x, grid_y])

# Merge free and occupied positions into a single list
grid_cells = grid_free_cells + grid_occ_cells
map_cells = map_free_cells + map_occ_cells
world_coords = world_free_coords + world_occ_coords

# Count the number of occupiable positions
num_free_cells = len(grid_free_cells)

In [6]:
# starting_grid_positions = []
# starting_orientations = []

# for _ in range(NUM_EPOCHS):
#     # Randomly select a free cell
#     starting_grid_positions.append(random.choice(grid_free_cells))
#     starting_orientations.append(random.choice([0, 90, 180, 270])) # Random orientation in degrees

# # Print it like it was a list
# print("starting_grid_positions = [", end="")
# for pos in starting_grid_positions:
#     print(f"{pos}, ", end="")
# print("]")

# print("starting_orientations = [", end="")
# for ori in starting_orientations:
#     print(f"{ori}, ", end="")
# print("]")

starting_grid_positions = [[45, 28], [15, 6], [40, 35], [25, 15], [6, 35], [9, 37], [31, 25], [21, 25], [50, 21], [24, 44], [41, 58], [48, 28], [22, 22], [13, 44], [23, 48], [37, 38], [47, 14], [29, 20], [27, 50], [15, 27], [19, 34], [12, 34], [31, 14], [16, 25], [35, 50], [27, 26], [7, 1], [17, 13], [16, 15], [39, 4], [43, 11], [21, 15], [17, 6], [47, 7], [45, 43], [16, 35], [7, 35], [28, 16], [10, 32], [12, 32], [50, 15], [14, 9], [47, 46], [39, 34], [24, 39], [36, 46], [23, 47], [46, 42], [25, 57], [12, 33], [6, 41], [40, 24], [8, 32], [24, 14], [37, 7], [22, 22], [42, 10], [8, 35], [5, 28], [35, 42], [9, 15], [49, 24], [4, 28], [13, 54], [22, 3], [49, 31], [26, 25], [42, 26], [24, 29], [12, 19], [20, 7], [24, 34], [38, 43], [26, 49], [49, 23], [29, 40], [18, 7], [25, 23], [40, 29], [26, 7], [26, 34], [43, 51], [15, 28], [10, 5], [1, 29], [18, 21], [7, 25], [25, 24], [18, 5], [36, 37], [9, 29], [11, 35], [50, 38], [33, 34], [22, 34], [38, 23], [26, 32], [50, 22], [22, 37], [38, 50],

In [7]:
starting_grid_positions = [[45, 28], [15, 6], [40, 35], [25, 15], [6, 35], [9, 37], [31, 25], [21, 25], [50, 21], [24, 44], [41, 58], [48, 28], [22, 22], [13, 44], [23, 48], [37, 38], [47, 14], [29, 20], [27, 50], [15, 27], [19, 34], [12, 34], [31, 14], [16, 25], [35, 50], [27, 26], [7, 1], [17, 13], [16, 15], [39, 4], [43, 11], [21, 15], [17, 6], [47, 7], [45, 43], [16, 35], [7, 35], [28, 16], [10, 32], [12, 32], [50, 15], [14, 9], [47, 46], [39, 34], [24, 39], [36, 46], [23, 47], [46, 42], [25, 57], [12, 33], [6, 41], [40, 24], [8, 32], [24, 14], [37, 7], [22, 22], [42, 10], [8, 35], [5, 28], [35, 42], [9, 15], [49, 24], [4, 28], [13, 54], [22, 3], [49, 31], [26, 25], [42, 26], [24, 29], [12, 19], [20, 7], [24, 34], [38, 43], [26, 49], [49, 23], [29, 40], [18, 7], [25, 23], [40, 29], [26, 7], [26, 34], [43, 51], [15, 28], [10, 5], [1, 29], [18, 21], [7, 25], [25, 24], [18, 5], [36, 37], [9, 29], [11, 35], [50, 38], [33, 34], [22, 34], [38, 23], [26, 32], [50, 22], [22, 37], [38, 50], ]
starting_orientations = [0, 180, 0, 90, 180, 180, 90, 270, 270, 0, 270, 90, 270, 270, 180, 90, 90, 270, 270, 90, 180, 180, 0, 0, 0, 180, 180, 90, 180, 0, 90, 180, 0, 0, 180, 0, 0, 180, 0, 90, 180, 270, 180, 90, 270, 180, 270, 270, 0, 270, 270, 90, 0, 0, 270, 90, 270, 180, 270, 90, 90, 270, 180, 0, 270, 90, 180, 90, 270, 90, 90, 180, 180, 90, 90, 90, 180, 270, 180, 0, 90, 90, 180, 90, 180, 270, 270, 270, 270, 180, 90, 270, 270, 90, 270, 180, 90, 180, 0, 180, ]

In [8]:
# Epochs metrics
num_actions_epochs = []
travelled_distance_epochs = []
success_epochs = []
location_error_epochs = []

In [9]:
for index, (GRID_POSITION, AGENT_YAW) in enumerate(zip(starting_grid_positions, starting_orientations)):

    # Initialize belief map: NUM_CLASSES + 1 for the background 
    belief_map = [[np.ones(NUM_CLASSES + 1) * DIRICHLET_PRIOR for _ in range(grid_resolution[1])] for _ in range(grid_resolution[0])]

    # Initialize an agent
    agent = sim.initialize_agent(sim_settings["default_agent"])

    # Sample a random position (within the possible ones)
    grid_position = GRID_POSITION
    idx = grid_free_cells.index(grid_position)
    world_position = world_free_coords[idx]
    map_position = map_free_cells[idx]

    # Sample a random yaw rotation
    agent_yaw = AGENT_YAW
    agent_quart = myutils.yaw_to_quaternion(agent_yaw)

    # Set agent state
    agent_state = habitat_sim.AgentState()
    agent_state.position = world_position
    agent_state.rotation = agent_quart
    agent.set_state(agent_state)

    # Compute agent radius in both maps
    min_bounds, max_bounds = sim.pathfinder.get_bounds()
    x_dim = max_bounds[0] - min_bounds[0]
    topdown_radius = (AGENT_RADIUS / x_dim * topdown_resolution[0])
    grid_radius = (AGENT_RADIUS / x_dim * grid_resolution[0])

    # Get initial agent position tuple and radius tuple
    agent_radius = (topdown_radius, grid_radius)
    agent_positions = (map_position, grid_position)

    # Metrics init
    target_found = False
    num_actions = 0
    travelled_distance = 0.0
    location_error = float("inf")

    # Simulation parameters
    num_clusters = simtools.num_cluster_centers(num_free_cells, max_clusters=num_free_cells)
    w_e, w_d, w_p = 0.4, 0.4, 0.1  # Weights for entropy, distance, and target probability
    MAX_ITER = int(num_free_cells * MAX_ITER_COEF)  # Maximum number of actions to perform

    # Print simulation 
    print(f"\nSimulation {index+1}/{len(starting_grid_positions)}")

    # Main simulation loop
    while (num_clusters <= num_free_cells) and (num_actions < MAX_ITER) and (not target_found):
        
        # Cluster grid map
        cluster_map = simtools.cluster_mapping(grid_free_cells, num_clusters)
        cluster_centers = simtools.get_cluster_centers(cluster_map, num_clusters)
        centroids = cluster_centers.copy() # For displaying logic
        cluster_occ_cells_map = simtools.assign_cluster_centers_to_cells(grid_occ_cells, cluster_centers, grid_free_cells)

        # Entropy map
        entropy_map = simtools.compute_entropy_map(belief_map, grid_map)

        # Display the simulation state
        #simtools.display_topdown_and_entropy_maps(topdown_map, grid_map, entropy_map, cluster_map, centroids, agent_positions, agent_radius, agent_yaw)

        # Move into each cluster center
        while (cluster_centers) and (num_actions < MAX_ITER) and (not target_found):

            # Computes scores for each cluster center
            distances, avg_entropies, avg_target_probs = [], [], []
            for cluster_center in cluster_centers:
                # Path distance 
                distance, _ = simtools.compute_path(grid_position, cluster_center, grid_free_cells)
                distances.append(distance)

                # Average entropy of the cluster
                avg_entropy = simtools.compute_average_entropy(belief_map, cluster_occ_cells_map[tuple(cluster_center)])
                avg_entropies.append(avg_entropy)

                # Target probability in the cluster surroundings
                avg_target_prob = simtools.compute_max_target_probability(belief_map, cluster_occ_cells_map[tuple(cluster_center)], indoor_objects.index(TARGET_OBJECT))
                avg_target_probs.append(avg_target_prob)

            # Normalize score components
            distances, avg_entropies, avg_target_probs = np.array(distances), np.array(avg_entropies), np.array(avg_target_probs)
            distances = (distances-np.min(distances)) / (np.max(distances)-np.min(distances)) if np.max(distances) != np.min(distances) else np.zeros_like(distances)
            avg_entropies = (avg_entropies-np.min(avg_entropies)) / (np.max(avg_entropies)-np.min(avg_entropies)) if abs(np.max(avg_entropies) - np.min(avg_entropies)) > 1e-5 else np.zeros_like(avg_entropies)
            avg_target_probs = (avg_target_probs-np.min(avg_target_probs)) / (np.max(avg_target_probs)-np.min(avg_target_probs)) if np.max(avg_target_probs) != np.min(avg_target_probs) else np.zeros_like(avg_target_probs)

            # Select best cluster center based on the scores
            scores = w_e * avg_entropies + w_d * (1-distances) + w_p * avg_target_probs
            max_score_idx = np.argmax(scores)
            cluster_center = cluster_centers[max_score_idx]
            _, path = simtools.compute_path(grid_position, cluster_center, grid_free_cells)

            # Move through the path
            while (path) and (num_actions < MAX_ITER) and (not target_found):
                
                # Compute actions to move to next path cell
                action_list = simtools.compute_relative_actions(grid_position, agent_yaw, path[0], ACTIONS)

                # Perform the rotation action, if needed
                for action in action_list:

                    # Perform action
                    grid_position, agent_yaw = simtools.perform_action(action, grid_position, agent_yaw)
                    idx = grid_free_cells.index(grid_position)
                    map_position, world_position = map_free_cells[idx], world_free_coords[idx]
                    agent_positions = (map_position, grid_position)
                    agent_quart = myutils.yaw_to_quaternion(agent_yaw)

                    # Update metrics
                    num_actions += 1
                    travelled_distance += simtools.compute_travelled_distance(agent_state.position, world_position)

                    # Update agent state
                    agent_state.position = world_position
                    agent_state.rotation = agent_quart
                    agent.set_state(agent_state)

                    # Get observations
                    obs = sim.get_sensor_observations(0)
                    rgb, depth = obs["color_sensor"], obs["depth_sensor"]

                    # YOLO Prediction
                    results = yolo_model.predict(source=rgb[:,:,:3], device='cuda:0', conf=0.30, iou=0.40, verbose=False, max_det=10)
                    detections = simtools.parse_yolo_detections(results)
                    simtools.merge_rgb_yolo_outputs(rgb, detections)
                    target_found, target_bbox = simtools.was_target_found(TARGET_OBJECT_ID, detections, CONFIDENCE_THRESHOLD)
                    
                    # Initialize processed cells
                    processed_cells = []

                    # Process the observations
                    for det in detections:
                        box, class_id, confidence, name, prob_vector = itemgetter('box', 'class_id', 'confidence', 'name', 'prob_vector')(det)
                        scale = myutils.compute_bbox_scale(box, rgb)

                        # Center of bbox and depth value
                        center_x, center_y = simtools.get_box_center(box)
                        depth_value = depth[center_y, center_x]

                        # Project to real world, grid and map coords
                        camera_world_position = simtools.get_camera_pos_from_agent_pos(world_position, AGENT_HEIGHT)
                        object_position = simtools.compute_real_world_position_from_pixel(camera_world_position, agent_quart, depth_value, center_x, center_y, intrinsics)
                        object_map_position, object_grid_position = simtools.get_2d_coords(object_position, topdown_resolution, grid_resolution, sim.pathfinder)

                        # Check if projected cell is outside free grid cells
                        if object_grid_position in grid_free_cells:
                            object_grid_position = simtools.get_closest_grey_cell(tuple(object_grid_position), grid_map)

                        # Add this cell to processed cells
                        if tuple(object_grid_position) not in processed_cells:
                            processed_cells.append(tuple(object_grid_position))

                        # Likelihood vector: likelihood of each class (Rule 1)
                        likelihood_vector = probtools.compute_likelihood_vector(prob_vector, scale, dirichlet_priors, classes_bins)

                        # Kaplan Update to the belief map in that cell
                        grid_x, grid_y = object_grid_position
                        belief_map[grid_x][grid_y] = probtools.kaplan_update(belief_map[grid_x][grid_y], likelihood_vector)

                    # Compute every theoretically visible occupied cell
                    rays = simtools.simulate_visibility_rays(grid_map, grid_position, agent_yaw)
                    visible_occ_cells = simtools.compute_visible_occ_cells(rays, grid_map, depth, grid_cells, world_coords, grid_position, agent_state.rotation, intrinsics)
                
                    # Go through every visible occupied cell and update the belief map
                    for cell in visible_occ_cells:
                        if tuple(cell) not in processed_cells:

                            # Compute distance to the cell
                            grid_x, grid_y = cell
                            distance = np.linalg.norm((np.array(grid_position) - np.array([grid_x, grid_y]))* CELL_SIDE)

                            # Compute the likelihood vector for the cell (Rule 2)
                            likelihood_vector = probtools.compute_background_likelihood_vector(distance, NUM_CLASSES)

                            # Kaplan update to the belief map in that cell
                            belief_map[grid_x][grid_y] = probtools.kaplan_update(belief_map[grid_x][grid_y], likelihood_vector)

                    # Entropy map
                    entropy_map = simtools.compute_entropy_map(belief_map, grid_map)

                    # Check if target was found
                    if target_found: break

                # Remove the path cell
                path.pop(0)

            # Check if target was found
            if target_found: break

            # When in cluster center, rotate 3 times to get the full 360 degrees view
            #simtools.display_sim_observations(rgb, depth)
            #simtools.display_topdown_and_entropy_maps(topdown_map, grid_map, entropy_map, cluster_map, centroids, agent_positions, agent_radius, agent_yaw)
            for i in range(3):
                grid_position, agent_yaw = simtools.perform_action('turn_right', grid_position, agent_yaw)
                agent_quart = myutils.yaw_to_quaternion(agent_yaw)
                agent_state.rotation = agent_quart
                agent.set_state(agent_state)

                # Update metrics
                num_actions += 1

                # Get observations
                observations = sim.get_sensor_observations(0)
                rgb = observations["color_sensor"]
                depth = observations["depth_sensor"]

                # YOLO Prediction
                results = yolo_model.predict(source=rgb[:,:,:3], device='cuda:0', conf=0.30, iou=0.40, verbose=False, max_det=10)
                detections = simtools.parse_yolo_detections(results)
                simtools.merge_rgb_yolo_outputs(rgb, detections)
                target_found, target_bbox = simtools.was_target_found(TARGET_OBJECT_ID, detections, CONFIDENCE_THRESHOLD)

                # Initialize processed cells
                processed_cells = []

                # Process the observations
                for det in detections:
                    box, class_id, confidence, name, prob_vector = itemgetter('box', 'class_id', 'confidence', 'name', 'prob_vector')(det)
                    scale = myutils.compute_bbox_scale(box, rgb)

                    # Center of bbox and depth value
                    center_x, center_y = simtools.get_box_center(box)
                    depth_value = depth[center_y, center_x]

                    # Project to real world, grid and map coords
                    camera_world_position = simtools.get_camera_pos_from_agent_pos(world_position, AGENT_HEIGHT)
                    object_position = simtools.compute_real_world_position_from_pixel(camera_world_position, agent_quart, depth_value, center_x, center_y, intrinsics)
                    object_map_position, object_grid_position = simtools.get_2d_coords(object_position, topdown_resolution, grid_resolution, sim.pathfinder)

                    # Check if projected cell is outside free grid cells
                    if object_grid_position in grid_free_cells:
                        object_grid_position = simtools.get_closest_grey_cell(tuple(object_grid_position), grid_map)

                    # Add this cell to processed cells
                    if tuple(object_grid_position) not in processed_cells:
                        processed_cells.append(tuple(object_grid_position))

                    # Likelihood vector: likelihood of each class (Rule 1)
                    likelihood_vector = probtools.compute_likelihood_vector(prob_vector, scale, dirichlet_priors, classes_bins)

                    # Kaplan Update to the belief map in that cell
                    grid_x, grid_y = object_grid_position
                    belief_map[grid_x][grid_y] = probtools.kaplan_update(belief_map[grid_x][grid_y], likelihood_vector)

                # Compute every theoretically visible occupied cell
                rays = simtools.simulate_visibility_rays(grid_map, grid_position, agent_yaw)
                visible_occ_cells = simtools.compute_visible_occ_cells(rays, grid_map, depth, grid_cells, world_coords, grid_position, agent_state.rotation, intrinsics)

                # Go through every visible occupied cell and update the belief map
                for cell in visible_occ_cells:
                    if tuple(cell) not in processed_cells:

                        # Compute distance to the cell
                        grid_x, grid_y = cell
                        distance = np.linalg.norm((np.array(grid_position) - np.array([grid_x, grid_y]))* CELL_SIDE)

                        # Compute the likelihood vector for the cell (Rule 2)
                        likelihood_vector = probtools.compute_background_likelihood_vector(distance, NUM_CLASSES)

                        # Kaplan update to the belief map in that cell
                        belief_map[grid_x][grid_y] = probtools.kaplan_update(belief_map[grid_x][grid_y], likelihood_vector)

                # Entropy map
                entropy_map = simtools.compute_entropy_map(belief_map, grid_map)

                # Check if target was found
                if target_found: break 

            # Remove cluster center
            cluster_centers.remove(cluster_center)

        # Double the number of clusters if all clusters were visited
        num_clusters *= 2

    # Target found or not
    if target_found:
        # Real world position 
        center_x, center_y = simtools.get_box_center(target_bbox)
        depth_value = depth[center_y, center_x]
        target_object_position = simtools.compute_real_world_position_from_pixel(agent_state.position, agent_state.rotation, depth_value, center_x, center_y, intrinsics)

        # Location_error to the target location
        location_error = simtools.compute_location_error(target_object_position, REAL_TARGET_LOCATION)

        # Check if location is valid
        if location_error > LOCATION_ERROR_THRESHOLD:
            print(f"\nTarget object <{TARGET_OBJECT}> found after {num_actions} actions, but location error {location_error:.3f} m exceeds threshold {LOCATION_ERROR_THRESHOLD} m. FAILURE!")
            target_found = False
            location_error = float("inf")
    else:
        # At this point, if target was not found, it might have been observed but without enough confidence. Check the target probability in the entropy map.
        target_found, target_grid_position = simtools.check_target_probability_in_entropy_map(belief_map, grid_cells, indoor_objects.index(TARGET_OBJECT), PSEUDO_COUNT_THRESHOLD)

        if target_found:
            # Convert target grid position to real world coordinates
            target_object_position = world_coords[grid_cells.index(target_grid_position)]
            target_map_position = map_cells[grid_cells.index(target_grid_position)]

            # Location_error to the target location
            location_error = simtools.compute_location_error(target_object_position, REAL_TARGET_LOCATION)

            # Check if location is valid
            if location_error > LOCATION_ERROR_THRESHOLD:
                print(f"\nTarget object <{TARGET_OBJECT}> found after {num_actions} actions, but location error {location_error:.3f} m exceeds threshold {LOCATION_ERROR_THRESHOLD} m. FAILURE!")
                target_found = False
                location_error = float("inf")

                print(f"Found location: {target_object_position}")
                print(f"Real location: {REAL_TARGET_LOCATION}")

                target_real_map_position, target_real_grid_position = simtools.get_2d_coords(REAL_TARGET_LOCATION, topdown_resolution, grid_resolution, sim.pathfinder)
                target_2d_coords = (target_map_position, target_grid_position)
                target_real_2d_coords = (target_real_map_position, target_real_grid_position)
                target_coords = (target_real_2d_coords, target_2d_coords)

    if target_found:
        # Process data
        target_real_map_position, target_real_grid_position = simtools.get_2d_coords(REAL_TARGET_LOCATION, topdown_resolution, grid_resolution, sim.pathfinder)
        target_map_position, target_grid_position = simtools.get_2d_coords(target_object_position, topdown_resolution, grid_resolution, sim.pathfinder)
        target_2d_coords = (target_map_position, target_grid_position)
        target_real_2d_coords = (target_real_map_position, target_real_grid_position)
        target_coords = (target_real_2d_coords, target_2d_coords)

        # Print results
        print(f"\nTarget object <{TARGET_OBJECT}> found after {num_actions} actions!")
        print(f"Found location: {target_object_position}")
        print(f"Real location: {REAL_TARGET_LOCATION}")
    else:
        print(f"\nTarget object <{TARGET_OBJECT}> not found after {MAX_ITER} actions!")
        print(f"Real location: {REAL_TARGET_LOCATION}")

    # Display simulation metrics
    print(f"Number of actions: {num_actions}")
    print(f"Travelled distance: {travelled_distance:.2f} m")
    print(f"Computed localization error: {location_error:.3f} m")

    # Append metrics to epochs
    num_actions_epochs.append(num_actions)
    travelled_distance_epochs.append(travelled_distance)
    success_epochs.append(target_found)
    location_error_epochs.append(location_error)


Simulation 1/100

Target object <microwave> found after 211 actions!
Found location: [    -11.567    -0.12646      2.1886]
Real location: [-11.539888358199647, 0.9425563303598442, 2.0594434993852118]
Number of actions: 211
Travelled distance: 44.33 m
Computed localization error: 0.132 m

Simulation 2/100

Target object <microwave> found after 291 actions!
Found location: [    -11.534    -0.12646      2.1886]
Real location: [-11.539888358199647, 0.9425563303598442, 2.0594434993852118]
Number of actions: 291
Travelled distance: 62.44 m
Computed localization error: 0.129 m

Simulation 3/100

Target object <microwave> found after 917 actions!
Found location: [    -11.534    -0.12646      2.1886]
Real location: [-11.539888358199647, 0.9425563303598442, 2.0594434993852118]
Number of actions: 917
Travelled distance: 195.09 m
Computed localization error: 0.129 m

Simulation 4/100

Target object <microwave> found after 82 actions!
Found location: [    -11.516    -0.12462      2.1886]
Real loca

In [10]:
import numpy as np

# Pre-process
num_actions_epochs = np.array(num_actions_epochs)
travelled_distance_epochs = np.array(travelled_distance_epochs)
success_epochs = np.array(success_epochs)
location_error_epochs = np.array(location_error_epochs)

# Counting
num_total_epochs = len(num_actions_epochs)
num_success_epochs = np.sum(success_epochs)

# Metrics on all runs
total_avg_num_actions = np.sum(num_actions_epochs) / num_total_epochs
total_avg_tavelled_distance = np.sum(travelled_distance_epochs) / num_total_epochs
success_rate = num_success_epochs / num_total_epochs * 100

# Metrics on successful runs
success_avg_travelled_distance = np.sum(travelled_distance_epochs[success_epochs]) / num_success_epochs
success_avg_num_actions = np.sum(num_actions_epochs[success_epochs]) / num_success_epochs
success_avg_location_error = np.sum(location_error_epochs[success_epochs]) / num_success_epochs

#Print final metrics
print("\n\nMETRICS:\n")
print(f"Total number of epochs: {num_total_epochs}")
print(f"Number of successful epochs: {num_success_epochs}")
print(f"Success rate: {success_rate:.2f}%")
print(f"Average number of actions (epochs): {total_avg_num_actions:.2f}")
print(f"Average travelled distance (epochs): {total_avg_tavelled_distance:.2f} m")
print(f"Average success rate (epochs): {success_rate:.2f}%")
print(f"\nAverage number of actions (successful epochs): {success_avg_num_actions:.2f}")
print(f"Average travelled distance (successful epochs): {success_avg_travelled_distance:.2f} m")
print(f"Average location error (successful epochs): {success_avg_location_error:.3f} m")

# Number of actions max, min and std
print(f"\nMax number of actions (epochs): {np.max(num_actions_epochs)}")
print(f"Min number of actions (epochs): {np.min(num_actions_epochs)}")
print(f"Std number of actions (epochs): {np.std(num_actions_epochs)}")    



METRICS:

Total number of epochs: 100
Number of successful epochs: 100
Success rate: 100.00%
Average number of actions (epochs): 322.75
Average travelled distance (epochs): 69.09 m
Average success rate (epochs): 100.00%

Average number of actions (successful epochs): 322.75
Average travelled distance (successful epochs): 69.09 m
Average location error (successful epochs): 0.130 m

Max number of actions (epochs): 917
Min number of actions (epochs): 7
Std number of actions (epochs): 187.63748959096634


In [11]:
metrics_file = 'results/metrics.json'

# Load existing metrics if the file exists, otherwise start with an empty list
if os.path.exists(metrics_file):
    with open(metrics_file, 'r') as f:
        existing_metrics = json.load(f)
else:
    existing_metrics = []

# New metrics entry (convert numpy types to native Python types)
def to_python_type(val):
    if hasattr(val, "item"):
        return val.item()
    return val

new_metrics = {
    "simulation_index": to_python_type(INDEX),
    "scene": SCENE,
    "target_object": TARGET_OBJECT,
    "search_method": METHOD,
    "confidence_threshold": to_python_type(CONFIDENCE_THRESHOLD),
    "max_iter_coefficient": to_python_type(MAX_ITER_COEF),
    "location_error_threshold": to_python_type(LOCATION_ERROR_THRESHOLD),
    "pseudo_count_threshold": to_python_type(PSEUDO_COUNT_THRESHOLD),
    "num_total_epochs": to_python_type(num_total_epochs),
    "num_success_epochs": to_python_type(num_success_epochs),
    "success_rate": to_python_type(success_rate),
    "total_avg_num_actions": to_python_type(total_avg_num_actions),
    "total_avg_tavelled_distance": to_python_type(total_avg_tavelled_distance),
    "success_avg_num_actions": to_python_type(success_avg_num_actions),
    "success_avg_travelled_distance": to_python_type(success_avg_travelled_distance),
    "success_avg_location_error": to_python_type(success_avg_location_error),
}

# Check for duplicate (by simulation_index and search_method)
duplicate_exists = any(
    entry["simulation_index"] == new_metrics["simulation_index"] and
    entry["search_method"] == new_metrics["search_method"] and
    entry["confidence_threshold"] == new_metrics["confidence_threshold"] and 
    entry["max_iter_coefficient"] == new_metrics["max_iter_coefficient"] and
    entry["location_error_threshold"] == new_metrics["location_error_threshold"] and
    entry["pseudo_count_threshold"] == new_metrics["pseudo_count_threshold"]
    for entry in existing_metrics
)

if not duplicate_exists:
    existing_metrics.append(new_metrics)
else:
    print(f"Metrics for simulation_index {new_metrics['simulation_index']} and search_method '{new_metrics['search_method']}' already exist. Skipping append.")

# Save updated metrics list
with open(metrics_file, 'w') as f:
    json.dump(existing_metrics, f, indent=4)

print(f"\nMetrics saved to {metrics_file}")



Metrics saved to results/metrics.json


In [12]:
sim.close()